# 06 — Joint training v2: grid adaptation → physics test → fixed joint training

This notebook fixes every cause found in joint training v1 (`docs/mtech/HAZARD_EXPERIMENT.md`):

| v1 cause | Fix here |
|---|---|
| Encoder trained on the old padded grid | **Stage A**: adapt the encoder to the corrected 6×6 grid (weather ± physics), several seeds |
| Physics comparison confounded by the grid | **Stage B**: frozen hazard heads on the Stage-A encoders, physics vs no-physics per seed, bootstrap CIs |
| Uncertainty weighting rewarded overfitting hazards | **Stage C**: capped hazard weights, and **GradNorm** as a second balancer |
| 24 duplicated hourly samples per label | hazard losses use one sample per day (00 UTC) |
| Hazard heads overfit | lower encoder lr (3e-5), head dropout 0.3, weight decay 1e-3 |
| Selection ignored weather | best hazard epoch only among epochs whose validation weather is within 1 % of the start |
| fp16 NaN batch | non-finite batches are skipped and counted |

**Attach:** the corrected 6×6 dataset, `checkpoints_for_regional_forecasting_baseline` (λ=0), and the IMD rainfall.
**Accelerator:** GPU T4 ×2, Internet on. Run once with `SMOKE_TEST = True` (~15 min), then set it to `False` and *Save & Run All (Commit)*.
A full run is roughly 3 h.

In [ ]:
SMOKE_TEST = True
SEEDS = [0, 1, 2]                 # Stage A seeds (smoke uses only the first)
STAGE_A_EPOCHS = 10
STAGE_C_EPOCHS = 15
RUN_STAGE_A, RUN_STAGE_B, RUN_STAGE_C = True, True, True
STAGE_C_BALANCERS = ["uncertainty", "gradnorm"]
BATCH_SIZE = 128
DELETE_FEATURE_CACHE = True       # Stage B feature files are large and not needed afterwards

INPUT_ROOT = "/kaggle/input"
CORRECTED_DIR_OVERRIDE = None
IMD_DIR_OVERRIDE = None
NOPHYSICS_CKPT_OVERRIDE = None    # the λ=0 internship checkpoint used to start Stage A
REPO_URL = "https://github.com/Aditya1Agrawal/ClimaX-Monsoon-PINN.git"
REPO_BRANCH = "mtech-extreme-events"
WORK = "/kaggle/working"

In [ ]:
import os, sys, json, subprocess, pathlib, importlib.util, time, shutil
REPO_DIR = f"{WORK}/ClimaX-Monsoon-PINN"
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, REPO_DIR], check=True)
print("repo commit:", subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "timm==0.6.12"], check=True)
timm_init = pathlib.Path(importlib.util.find_spec("timm").origin).parent / "models" / "__init__.py"
txt = timm_init.read_text()
if sys.version_info >= (3, 11) and "\nfrom .maxxvit import *" in txt:
    timm_init.write_text(txt.replace("\nfrom .maxxvit import *", "\n# from .maxxvit import *  # disabled for py>=3.11"))
sys.path.insert(0, f"{REPO_DIR}/src")
import torch, numpy as np, pandas as pd
print("torch", torch.__version__, "| GPUs:", torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

## Inputs

In [ ]:
from climax_hazards import inspect_utils as iu
cands = [p for p in iu.find_dataset_dirs(INPUT_ROOT)
         if np.load(p / "lon.npy").size == 6 and not iu.detect_lon_padding(p)["looks_padded"]]
CORR_DIR = str(iu.resolve_single(cands, CORRECTED_DIR_OVERRIDE, "corrected 6x6 dataset"))
def find_imd(root):
    for dirpath, _, files in os.walk(root):
        if "1979.grd" in files:
            return dirpath
IMD_DIR = IMD_DIR_OVERRIDE or find_imd(INPUT_ROOT)
print("corrected dataset:", CORR_DIR, "\nIMD:", IMD_DIR or "NOT FOUND - rainfall skipped")

rows = []
for p in iu.find_checkpoints(INPUT_ROOT):
    ck = iu.load_checkpoint(p)
    mc = next((v for k, v in (ck.get("callbacks") or {}).items() if k.startswith("ModelCheckpoint")), {})
    rows.append({"path": str(p), "run_dir": str(p.parent), "file": p.name, "lambda": ck.get("hyper_parameters", {}).get("physics_lambda"),
                 "epoch": ck.get("epoch"), "best_at_save": os.path.basename(mc.get("best_model_path", "") or "")})
    del ck
ckpts = pd.DataFrame(rows)
final = ckpts.sort_values("epoch").groupby("run_dir").tail(1).set_index("run_dir")
ckpts["run_final_best"] = [r.file == final.loc[r.run_dir, "best_at_save"] for r in ckpts.itertuples()]
print(ckpts.drop(columns=["path", "run_dir"]).to_string(index=False))
if NOPHYSICS_CKPT_OVERRIDE:
    INIT = NOPHYSICS_CKPT_OVERRIDE
else:
    sel = ckpts[(ckpts["lambda"] == 0.0) & ckpts["run_final_best"]]
    assert len(sel) == 1, f"expected one final-best λ=0 checkpoint, found {len(sel)}; set NOPHYSICS_CKPT_OVERRIDE"
    INIT = sel["path"].iloc[0]
print("\nStage A starts from (λ=0):", INIT)

In [ ]:
WORK = os.path.abspath(WORK)
env = dict(os.environ, PYTHONPATH=f"{REPO_DIR}/src")
tag = "_smoke" if SMOKE_TEST else ""
seeds = SEEDS[:1] if SMOKE_TEST else SEEDS
def run(cmd, log_path):
    t0 = time.time()
    with open(log_path, "w") as log:
        proc = subprocess.Popen(cmd, cwd=REPO_DIR, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            log.write(line)
            if not line.startswith(" ") or "seed" in line:   # keep the notebook log readable
                print(line, end="", flush=True)
        rc = proc.wait()
    print(f"--> exit {rc} after {(time.time() - t0) / 60:.1f} min")
    assert rc == 0, f"failed, see {log_path}"
SUMMARY = f"{WORK}/summary{tag}"
os.makedirs(SUMMARY, exist_ok=True)

## Stage A — adapt the encoder to the corrected grid (weather ± physics, several seeds)

In [ ]:
A = {}
for variant in ("phys", "nophys"):
    for seed in seeds:
        name = f"A_{variant}_s{seed}"
        out = f"{WORK}/stageA/{name}{tag}"
        A[name] = out
        if not RUN_STAGE_A or os.path.exists(f"{out}/best_encoder.ckpt"):
            continue
        os.makedirs(out, exist_ok=True)
        cmd = [sys.executable, "scripts/mtech/train_joint.py", "--data_dir", CORR_DIR, "--init_ckpt", INIT, "--out_dir", out,
               "--no_hazards", "--export_encoder", "--balancing", "uncertainty", "--seed", str(seed), "--epochs", str(STAGE_A_EPOCHS),
               "--batch_size", str(BATCH_SIZE),
               "--patience", "3"]
        if variant == "nophys":
            cmd += ["--no_physics"]
        if SMOKE_TEST:
            cmd += ["--smoke"]
        print(f"\n===== {name} =====")
        run(cmd, f"{out}/run.log")

In [ ]:
key_vars = ["2m_temperature", "10m_u_component_of_wind", "temperature_850", "geopotential_500", "geopotential_850",
            "hyps_resid_500_850", "hyps_resid_700_850", "hyps_resid_850_925"]
rowsA = []
for name, out in A.items():
    wt = pd.read_csv(f"{out}/weather_test.csv")
    meta = json.load(open(f"{out}/run_meta.json"))
    hist = pd.read_csv(f"{out}/history.csv")
    best_val = hist[hist.epoch >= 0]["val_w_mse"].min()
    for _, r in wt.iterrows():
        rowsA.append({"run": name, "variant": name.split("_")[1], "seed": int(name.split("_s")[-1]), "source": r["source"],
                      "variable": r["variable"], "rmse": r["rmse"], "best_val_w_mse": best_val, "selected_epoch": meta["selected_epoch"]})
dfA = pd.DataFrame(rowsA)
dfA.to_csv(f"{SUMMARY}/stageA_weather_all.csv", index=False)
model_rows = dfA[dfA.source == "joint"]
tabA = model_rows[model_rows.variable.isin(key_vars)].groupby(["variable", "variant"])["rmse"].agg(["mean", "std", "count"]).unstack("variant")
clim = dfA[(dfA.source == "climatology") & dfA.variable.isin(key_vars)].groupby("variable")["rmse"].first()
print("Stage A test RMSE, mean (sd) over seeds; climatology for reference:")
print(pd.concat([tabA, clim.rename("climatology")], axis=1).round(3).to_string())
tabA.to_csv(f"{SUMMARY}/stageA_weather_table.csv")
print("\nbest validation w_mse per run:\n", dfA.groupby("run")[["best_val_w_mse", "selected_epoch"]].first().round(4).to_string())

## Stage B — frozen hazard heads on the Stage-A encoders (physics vs no-physics on the correct grid)

In [ ]:
FROZEN_B = f"{WORK}/stageB{tag}"
if RUN_STAGE_B and not os.path.exists(f"{FROZEN_B}/hazard_summary_test.csv"):
    os.makedirs(FROZEN_B, exist_ok=True)
    cmd = [sys.executable, "scripts/mtech/run_hazard_experiment.py", "--data_dir", CORR_DIR, "--out_dir", FROZEN_B]
    for name, out in A.items():
        cmd += ["--checkpoint", f"{name}={out}/best_encoder.ckpt"]
    for seed in seeds:
        cmd += ["--compare", f"A_phys_s{seed}:A_nophys_s{seed}"]
    if IMD_DIR:
        cmd += ["--imd_dir", IMD_DIR]
    if SMOKE_TEST:
        cmd += ["--max_samples", "300", "--epochs", "2", "--seeds", "0", "--no_logistic"]
    run(cmd, f"{FROZEN_B}/run.log")
    if DELETE_FEATURE_CACHE:
        shutil.rmtree(f"{FROZEN_B}/features", ignore_errors=True)

sB = pd.read_csv(f"{FROZEN_B}/hazard_summary_test.csv")
sB["variant"] = sB["source"].map(lambda s: s.split("_")[1] if s.startswith("A_") else s)
tabB = sB.groupby(["hazard", "variant"])["pr_auc"].agg(["mean", "std", "count"]).round(4)
print("Stage B test PR-AUC (frozen heads), mean (sd) over encoder seeds:")
print(tabB.to_string())
tabB.to_csv(f"{SUMMARY}/stageB_hazard_table.csv")
if os.path.exists(f"{FROZEN_B}/bootstrap_test.csv"):
    bB = pd.read_csv(f"{FROZEN_B}/bootstrap_test.csv")
    print("\nphysics − no-physics per seed (paired 7-day block bootstrap, 95% CI):")
    print(bB.round(4).to_string(index=False))
    bB.to_csv(f"{SUMMARY}/stageB_bootstrap.csv", index=False)

## Stage C — joint training v2 from the best Stage-A physics encoder

In [ ]:
hist_best = {name: pd.read_csv(f"{out}/history.csv").query("epoch >= 0")["val_w_mse"].min() for name, out in A.items() if "_phys_" in name}
BEST_A = min(hist_best, key=hist_best.get)
print("Stage C starts from", BEST_A, f"(best validation w_mse {hist_best[BEST_A]:.4f})")
C = {}
for bal in STAGE_C_BALANCERS:
    name = f"C_{bal}"
    out = f"{WORK}/stageC/{name}{tag}"
    C[name] = out
    if not RUN_STAGE_C or os.path.exists(f"{out}/run_meta.json"):
        continue
    os.makedirs(out, exist_ok=True)
    srcs = [s for s in (BEST_A, BEST_A.replace("_phys_", "_nophys_"), "logistic", "climatology", "persistence")
            if os.path.exists(f"{FROZEN_B}/preds/{s}.npz")]
    cmd = [sys.executable, "scripts/mtech/train_joint.py", "--data_dir", CORR_DIR, "--out_dir", out,
           "--init_ckpt", f"{A[BEST_A]}/best_encoder.ckpt", "--balancing", bal, "--epochs", str(STAGE_C_EPOCHS), "--batch_size", str(BATCH_SIZE),
           "--lr_encoder", "3e-5", "--head_dropout", "0.3", "--head_weight_decay", "1e-3", "--patience", "5",
           "--hazard_hours", "0", "--weather_tolerance", "0.01", "--compare_dir", FROZEN_B, "--compare_sources", *srcs]
    if bal == "uncertainty":
        cmd += ["--max_hazard_weight", "2"]
    if IMD_DIR:
        cmd += ["--imd_dir", IMD_DIR]
    if SMOKE_TEST:
        cmd += ["--smoke"]
    print(f"\n===== {name} =====")
    run(cmd, f"{out}/run.log")

## Results summary

In [ ]:
rows = []
for name, out in C.items():
    hz = pd.read_csv(f"{out}/hazard_results.csv")
    for _, r in hz[hz.split == "test"].iterrows():
        rows.append({"source": name, "hazard": r["hazard"], "pr_auc": r["pr_auc"], "roc_auc": r["roc_auc"], "f1": r["f1"]})
for _, r in sB[sB.source.isin([BEST_A, BEST_A.replace("_phys_", "_nophys_"), "logistic", "climatology", "persistence"])].iterrows():
    rows.append({"source": r["source"] + ("" if r["source"] in ("logistic", "climatology", "persistence") else " (frozen)"),
                 "hazard": r["hazard"], "pr_auc": r["pr_auc"], "roc_auc": r["roc_auc"], "f1": r["f1"]})
tabC = pd.DataFrame(rows).pivot_table(index="source", columns="hazard", values="pr_auc").round(4)
print("TEST PR-AUC (Stage C joint vs frozen heads on the same encoder and baselines):")
print(tabC.to_string())
tabC.to_csv(f"{SUMMARY}/stageC_hazard_table.csv")

for name, out in C.items():
    print(f"\n--- {name} ---")
    meta = json.load(open(f"{out}/run_meta.json"))
    print("selected epoch", meta["selected_epoch"], "|", meta["selection_rule"], "| skipped batches", meta["skipped_batches_total"])
    print("task weights at the selected epoch:", {k: round(v, 3) for k, v in meta["selected_task_weights"].items()})
    if os.path.exists(f"{out}/bootstrap_vs_frozen.csv"):
        print(pd.read_csv(f"{out}/bootstrap_vs_frozen.csv")[["b", "hazard", "pr_auc_diff", "ci_low", "ci_high"]].round(4).to_string(index=False))
    wt = pd.read_csv(f"{out}/weather_test.csv")
    joint_w = wt[wt.source == "joint"].set_index("variable")["rmse"]
    a_w = pd.read_csv(f"{A[BEST_A]}/weather_test.csv").query("source == 'joint'").set_index("variable")["rmse"]
    cl_w = wt[wt.source == "climatology"].set_index("variable")["rmse"]
    cmp = pd.DataFrame({"stageC": joint_w, "stageA_start": a_w, "climatology": cl_w}).loc[key_vars]
    cmp["change_vs_start_%"] = 100 * (cmp["stageC"] / cmp["stageA_start"] - 1)
    print(cmp.round(3).to_string())
    cmp.to_csv(f"{SUMMARY}/stageC_weather_{name}.csv")
print("\nsummary files:", sorted(os.listdir(SUMMARY)))